In [10]:
import torch
from torch import nn

In [47]:
# LSTM Model
class ModelLSTM(nn.Module):
    def __init__(self, dataset):
        super(ModelLSTM, self).__init__()
        self.lstm_size = 128
        self.embedding_dim = 128
        self.num_layers = 3

        n_vocab = len(dataset.uniq_words)
        self.embedding = nn.Embedding(num_embeddings=n_vocab, embedding_dim=self.embedding_dim)
        self.lstm = nn.LSTM(input_size=self.lstm_size, hidden_size=self.lstm_size, num_layers=self.num_layers, dropout=0.2)
        self.fc = nn.Linear(self.lstm_size, n_vocab)

    def forward(self, x, prev_state):
        embed = self.embedding(x)
        output, state = self.lstm(embed, prev_state)
        logits = self.fc(output)
        return logits, state

    def init_state(self, sequence_length):
        return (torch.zeros(self.num_layers, sequence_length, self.lstm_size),
                torch.zeros(self.num_layers, sequence_length, self.lstm_size))

In [81]:
class ModelGRU(nn.Module):
    def __init__(self, dataset):
        super(ModelGRU, self).__init__()
        self.gru_size = 128
        self.embedding_dim = 128
        self.num_layers = 3

        n_vocab = len(dataset.uniq_words)
        self.embedding = nn.Embedding(num_embeddings=n_vocab, embedding_dim=self.embedding_dim)
        self.gru = nn.GRU(input_size=self.gru_size, hidden_size=self.gru_size, num_layers=self.num_layers, dropout=0.2)
        self.fc = nn.Linear(self.gru_size, n_vocab)

    def forward(self, x, prev_state):
        embed = self.embedding(x)
        output, state = self.gru(embed, prev_state)
        logits = self.fc(output)
        return logits, state
    
    def init_state(self, sequence_length):
        return (torch.zeros(self.num_layers, sequence_length, self.gru_size),
                torch.zeros(self.num_layers, sequence_length, self.gru_size))

In [12]:
import torch
import pandas as pd
from collections import Counter

In [13]:
# Dataset Load
class Dataset(torch.utils.data.Dataset):
    def __init__(
        self,
        args,
    ):
        self.args = args
        self.words = self.load_words()
        self.uniq_words = self.get_uniq_words()

        self.index_to_word = {index: word for index, word in enumerate(self.uniq_words)}
        self.word_to_index = {word: index for index, word in enumerate(self.uniq_words)}

        self.words_indexes = [self.word_to_index[w] for w in self.words]

    def load_words(self):
        train_df = pd.read_csv('./Jokes.csv')
        text = train_df['Joke'].str.cat(sep=' ')
        return text.split(' ')

    def get_uniq_words(self):
        word_counts = Counter(self.words)
        return sorted(word_counts, key=word_counts.get, reverse=True)

    def __len__(self):
        return len(self.words_indexes) - self.args.sequence_length

    def __getitem__(self, index):
        return (
            torch.tensor(self.words_indexes[index:index+self.args.sequence_length]),
            torch.tensor(self.words_indexes[index+1:index+self.args.sequence_length+1]),
        )

In [14]:
import argparse
import torch
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader
#from model import Model
#from dataset import Dataset

In [15]:
# Model Train
def train(dataset, model, args):
    model.train()

    dataloader = DataLoader(dataset, batch_size=args.batch_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(args.max_epochs):
        state_h, state_c = model.init_state(args.sequence_length)

        for batch, (x, y) in enumerate(dataloader):
            optimizer.zero_grad()

            y_pred, (state_h, state_c) = model(x, (state_h, state_c))
            loss = criterion(y_pred.transpose(1, 2), y)

            state_h = state_h.detach()
            state_c = state_c.detach()

            loss.backward()
            optimizer.step()

            print({ 'epoch': epoch, 'batch': batch, 'loss': loss.item() })

In [16]:
# Model Evaluation(Text Generation)
def predict(dataset, model, text, next_words=100):
    model.eval()

    words = text.split(' ')
    state_h, state_c = model.init_state(len(words))

    for i in range(0, next_words):
        x = torch.tensor([[dataset.word_to_index[w] for w in words[i:]]])
        y_pred, (state_h, state_c) = model(x, (state_h, state_c))

        last_word_logits = y_pred[0][-1]
        p = torch.nn.functional.softmax(last_word_logits, dim=0).detach().numpy()
        word_index = np.random.choice(len(last_word_logits), p=p)
        words.append(dataset.index_to_word[word_index])

    return words

In [82]:
# Seting Arguments
parser = argparse.ArgumentParser()
parser.add_argument('--max-epochs', type=int, default=2)
parser.add_argument('--batch-size', type=int, default=64)
parser.add_argument('--sequence-length', type=int, default=4)
args, unknown = parser.parse_known_args()
#args = parser.parse_args()
print(args)

Namespace(max_epochs=2, batch_size=64, sequence_length=4)


In [83]:
# Run Model and Train
dataset = Dataset(args)
model = ModelLSTM(dataset)
#model = ModelGRU(dataset) # Error ?
train(dataset, model, args)

In [50]:
# Show Prediction
words = predict(dataset, model, text='Knock knock. Whos there?')
print(words)

['Knock', 'knock.', 'Whos', 'there?', 'turkey', 'mumble-bee.', 'during', 'did', 'go', 'the', 'go', 'are', 'in', 'cold', 'duck', 'layoffs', 'you', 'hamburglar', 'thing', 'frog', 'ointment', 'casinos', 'Sauron', 'steal', 'new', 'dirty', 'he', 'told', "Bulls'.", 'snack', 'intestine.', '/r/askreddit', 'jammed', 'and', 'to', 'cross', 'rabbits', 'an', 'found', 'then', "you've", 'Pikachu', 'hear', 'Graaaaaaiiiins......', 'lost', 'wrong', 'call', 'Suspoonders!', 'out', 'What', 'I', '*Frank', 'turtle', 'took', 'the', 'did', 'nothing', 'deep.', 'cross', 'very', 'breakfast', 'the', "what's", 'to', 'change', 'nose', 'empty', 'and', 'a', "She's", 'And', 'Because', 'What', 'run', 'tutor', 'field.', 'kill', 'In', 'calendar...', 'boars', 'He', 'culture', 'other,', 'Thanks!', 'friends', 'I', 'new', 'she', 'in:', 'the', 'want', "Bulls'.", 'rope', 'What', 'barbed', 'ear.', 'time', '(Not', 'little', 'O', 'your', 'beef...', 'joke', 'an']
